<a href="https://colab.research.google.com/github/postnicov/MICpredictOnline/blob/main/ML_monstr_test1_colab_urls.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML chem drugs

Google Colab notebook version of the original R Markdown workflow, updated to download all required input files from the provided GitHub URLs.

In [1]:
needed <- c("readxl", "dplyr", "writexl")

to_install <- needed[!sapply(needed, requireNamespace, quietly = TRUE)]
if (length(to_install) > 0) {
  install.packages(to_install, repos = "https://cloud.r-project.org")
}

library(readxl)
library(dplyr, warn.conflicts = FALSE)  # suppress masking messages
library(writexl)

options(digits = 3)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [ ]:
urls <- c(
  blind_test_data = "https://github.com/postnicov/MICpredictOnline/raw/refs/heads/main/model/blind_test_data.xlsx",
  features = "https://github.com/postnicov/MICpredictOnline/raw/refs/heads/main/model/features.rds",
  model = "https://github.com/postnicov/MICpredictOnline/raw/refs/heads/main/model/model.rds",
  top_features = "https://github.com/postnicov/MICpredictOnline/raw/refs/heads/main/model/top_features.rds"
)

dest_files <- c(
  blind_test_data = "blind_test_data.xlsx",
  features = "features.rds",
  model = "model.rds",
  top_features = "top_features.rds"
)

for (nm in names(urls)) {
  download.file(urls[[nm]], destfile = dest_files[[nm]], mode = "wb")
}

file.info(dest_files)[, c("size", "mtime")]

,size,mtime
,<dbl>,<dttm>
blind_test_data.xlsx,78212,2026-06-14 12:27:49
features.rds,1259,2026-06-14 12:27:50
model.rds,14919,2026-06-14 12:27:51
top_features.rds,172,2026-06-14 12:27:51


In [ ]:
model <- readRDS("model.rds")
expected_feats <- readRDS("features.rds")
top_features <- readRDS("top_features.rds")

new_data <- read_xlsx("blind_test_data.xlsx", sheet = 1)
ids <- new_data$ID

new_aligned <- new_data %>%
  select(all_of(expected_feats)) %>%
  mutate(across(everything(), as.numeric))

pred_class <- predict(model, newdata = new_aligned, type = "class")

get_fragments <- function(row_data) {
  active_features <- names(row_data)[as.numeric(row_data) == 1]
  active_features <- intersect(active_features, top_features)

  if (length(active_features) == 0) {
    return("None")
  }

  paste(active_features, collapse = "; ")
}

associated_features <- sapply(
  seq_len(nrow(new_aligned)),
  function(i) get_fragments(new_aligned[i, ])
)

result_df <- data.frame(
  ID = ids,
  Predicted_Class = as.character(pred_class),
  Associated_PubChem = associated_features
)

write_xlsx(result_df, "blind_prediction_with_fragments.xlsx")
result_df

ID,Predicted_Class,Associated_PubChem
<chr>,<chr>,<chr>
105.mol,2,PubchemFP180; PubchemFP181; PubchemFP188; PubchemFP337; PubchemFP345; PubchemFP346; PubchemFP365; PubchemFP366; PubchemFP540
111.mol,2,PubchemFP180; PubchemFP181; PubchemFP188; PubchemFP337; PubchemFP345; PubchemFP346; PubchemFP365; PubchemFP366; PubchemFP540
117.mol,2,PubchemFP358; PubchemFP396; PubchemFP403; PubchemFP409; PubchemFP453; PubchemFP472; PubchemFP492; PubchemFP539; PubchemFP612; PubchemFP682; PubchemFP685
119.mol,2,PubchemFP358; PubchemFP396; PubchemFP403; PubchemFP409; PubchemFP453; PubchemFP472; PubchemFP492; PubchemFP539; PubchemFP612; PubchemFP682; PubchemFP685
121.mol,2,PubchemFP345; PubchemFP346; PubchemFP365; PubchemFP366; PubchemFP409; PubchemFP492; PubchemFP540; PubchemFP612; PubchemFP682; PubchemFP685
124.mol,2,PubchemFP358; PubchemFP396; PubchemFP403; PubchemFP409; PubchemFP453; PubchemFP472; PubchemFP497; PubchemFP566; PubchemFP602; PubchemFP659; PubchemFP691
145.mol,2,PubchemFP345; PubchemFP358; PubchemFP365; PubchemFP396; PubchemFP403; PubchemFP453; PubchemFP472; PubchemFP539; PubchemFP540
159.mol,2,PubchemFP180; PubchemFP181; PubchemFP345; PubchemFP358; PubchemFP365; PubchemFP396; PubchemFP403; PubchemFP453; PubchemFP472; PubchemFP539; PubchemFP540
16.mol,2,PubchemFP345; PubchemFP358; PubchemFP365; PubchemFP396; PubchemFP403; PubchemFP453; PubchemFP472; PubchemFP539; PubchemFP540
